In [1]:
!pip install evaluate seqeval datasets transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=fc788d7db917ce5568dd8dc05543869fb48b5991e7bebc7511954d3f5e365875
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


Import Library & Setup Awal

In [2]:
import torch
import numpy as np
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from datasets import load_dataset

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

PyTorch version: 2.11.0+cu128
CUDA Available: True


Load & Eksplorasi Dataset

In [5]:
from datasets import load_dataset, DatasetDict

raw_datasets_temp = load_dataset("json", data_files="dataset_rekrutmen_sintetis.json")

# Membelah 500 data menjadi 80% Train dan 20% Validation
split_data = raw_datasets_temp["train"].train_test_split(test_size=0.2, seed=42)

# Susun ulang ke dalam struktur DatasetDict agar sesuai standar Hugging Face
raw_datasets = DatasetDict({
    "train": split_data["train"],
    "validation": split_data["test"]
})

label_list = [
    "O",
    "B-NAMA", "I-NAMA",
    "B-INSTITUSI", "I-INSTITUSI",
    "B-KETERAMPILAN", "I-KETERAMPILAN",
    "B-JABATAN", "I-JABATAN"
]

id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

print("Jumlah Data Train:", len(raw_datasets["train"]))
print("Jumlah Data Validation:", len(raw_datasets["validation"]))
print("\nContoh Teks Mentah (Indeks 0):")
print(raw_datasets["train"][0]["tokens"])

Generating train split: 0 examples [00:00, ? examples/s]

Jumlah Data Train: 400
Jumlah Data Validation: 100

Contoh Teks Mentah (Indeks 0):
['Saya', 'Natalia', 'Lestari', 'memiliki', 'pengalaman', 'sebagai', 'Frontend', 'Developer', 'dengan', 'keahlian', 'utama', 'Machine', 'Learning']


Inisialisasi Tokenizer & Model Transformer

In [6]:
model_checkpoint = "indobenchmark/indobert-base-p2"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1
)

print(f"Model {model_checkpoint} berhasil dimuat dengan {len(label_list)} label klasifikasi.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[transformers] You passed `num_labels=9` which is incompatible to the `id2label` map of length `5`.


pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: indobenchmark/indobert-base-p2
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model indobenchmark/indobert-base-p2 berhasil dimuat dengan 9 label klasifikasi.


Fungsi Alignment Token & Eksekusi Mapping

In [7]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []

    # Looping melalui setiap baris data
    for i, ner_tag_list in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # Token spesial ([CLS], [SEP]) diberi label -100 agar diabaikan saat loss calculation
            if word_idx is None:
                label_ids.append(-100)
            # Beri label pada awal kata
            elif word_idx != previous_word_idx:
                # PERBAIKAN DI SINI: Ubah string (misal "B-NAMA") menjadi angka ID (misal 1)
                label_string = ner_tag_list[word_idx]
                label_ids.append(label2id[label_string])
            # Subword lanjutan diberi -100
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Mapping ke seluruh dataset
tokenized_datasets = raw_datasets.map(tokenize_and_align_labels, batched=True)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

print("\nContoh Token Setelah Alignment & Konversi ke ID (Indeks 0):")
tokens = tokenizer.convert_ids_to_tokens(tokenized_datasets["train"][0]["input_ids"])
labels = tokenized_datasets["train"][0]["labels"]
for token, label in zip(tokens, labels):
    print(f"{token:<15} : {label}")

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]


Contoh Token Setelah Alignment & Konversi ke ID (Indeks 0):
[CLS]           : -100
saya            : 0
natal           : 1
##ia            : -100
lestari         : 2
memiliki        : 0
pengalaman      : 0
sebagai         : 0
front           : 7
##end           : -100
developer       : 8
dengan          : 0
keahlian        : 0
utama           : 0
machine         : 5
learning        : 6
[SEP]           : -100


Setup Metrik Evaluasi

In [8]:
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

print("Metrik Seqeval berhasil dimuat.")

Metrik Seqeval berhasil dimuat.


Eksekusi Fine-Tuning

In [9]:
args = TrainingArguments(
    output_dir="./indobert-ner-rekrutmen",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    # Argumen 'tokenizer' dihapus di sini
    compute_metrics=compute_metrics
)

print("Memulai proses fine-tuning Transformer...")
trainer.train()

# Menyimpan model dan tokenizer secara eksplisit
trainer.save_model("indobert-ner-rekrutmen-final")
tokenizer.save_pretrained("indobert-ner-rekrutmen-final")
print("Selesai! Model dan Tokenizer berhasil disimpan di folder 'indobert-ner-rekrutmen-final'.")

Memulai proses fine-tuning Transformer...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.061403,0.893333,0.875817,0.884488,0.974576
2,No log,0.019014,0.973333,0.954248,0.963696,0.992296
3,No log,0.009294,0.990132,0.983660,0.986885,0.997689
4,No log,0.009127,0.993377,0.980392,0.986842,0.996918
5,No log,0.007882,0.993399,0.983660,0.988506,0.997689


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Selesai! Model dan Tokenizer berhasil disimpan di folder 'indobert-ner-rekrutmen-final'.


In [ ]:
!zip -r indobert-ner-rekrutmen-final.zip ./indobert-ner-rekrutmen-final

  adding: indobert-ner-rekrutmen-final/ (stored 0%)
  adding: indobert-ner-rekrutmen-final/tokenizer.json (deflated 71%)
  adding: indobert-ner-rekrutmen-final/tokenizer_config.json (deflated 43%)
  adding: indobert-ner-rekrutmen-final/model.safetensors (deflated 7%)
  adding: indobert-ner-rekrutmen-final/config.json (deflated 59%)
  adding: indobert-ner-rekrutmen-final/training_args.bin (deflated 53%)


In [10]:
!mkdir models
!mkdir modules

In [11]:
!mv indobert-ner-rekrutmen-final models/

In [12]:
!apt-get install tesseract-ocr-ind tesseract-ocr-eng
!pip install streamlit sentence-transformers paddlepaddle paddleocr pytesseract opencv-python
!npm install localtunnel

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr-eng is already the newest version (1:4.00~git30-7274cfa-1.1).
tesseract-ocr-eng set to manually installed.
The following NEW packages will be installed:
  tesseract-ocr-ind
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 537 kB of archives.
After this operation, 1,138 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-ind all 1:4.00~git30-7274cfa-1.1 [537 kB]
Fetched 537 kB in 0s (7,779 kB/s)
Selecting previously unselected package tesseract-ocr-ind.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-ind_1%3a4.00~git30-7274cfa-1.1_all.deb ...
Unpacking tesseract-ocr-ind (1:4.00~git30-7274cfa-1.1) ...
Setting up tesseract-ocr-ind (1:4.00~git30-7274cfa-1.1) ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 5.4 MB/s 

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
added 22 packages in 2s
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇

In [1]:
import urllib
print("Password/Endpoint IP kamu adalah:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))

Password/Endpoint IP kamu adalah: 34.50.165.41


In [ ]:
# Matikan proses lama yang masih nyangkut
!pkill -f streamlit
!pkill -f localtunnel

# Unduh Cloudflare Tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# Jalankan Streamlit di latar belakang
!nohup streamlit run app.py &

# Buka terowongan Cloudflare
!./cloudflared tunnel --url http://localhost:8501

nohup: appending output to 'nohup.out'
2026-06-06T14:48:14Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-06T14:48:14Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-06-06T14:48:17Z INF +--------------------------------------------------------------------------------------------+
2026-06-06T14:48:17Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-06-06T14:48:17Z INF |  https://fur-di